In [17]:
import os
import json
import cv2
import numpy as np
from pathlib import Path

def evaluate_single_image(img_100_path, img_0_path, img_eval_path, json_path=None):
    """
    Évalue une image corrigée en la comparant à l'originale (100%) et à la base défectueuse (0%).
    """

    # ==========================================
    # Fonctions utilitaires intégrées
    # ==========================================
    def read_json(p):
        p = Path(p)
        if not p.is_file():
            raise FileNotFoundError(f"Fichier introuvable : {p}")
        return json.loads(p.read_text(encoding="utf-8"))

    def process_json(data):
        indices = []
        for c in data:
            indices += c["x_coord"]
        return np.array(indices, dtype=int)

    def load_image(path):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Image introuvable : {path}")
        return cv2.imread(path, -1).astype(np.float32, copy=False)

    # ==========================================
    # Chargement des images
    # ==========================================
    gt = load_image(img_100_path)      # 100% Original (Ground Truth)
    base = load_image(img_0_path)      # 0% Défectueux (Base)
    res = load_image(img_eval_path)    # Image à évaluer

    if gt.shape != base.shape or gt.shape != res.shape:
        raise ValueError("Les trois images doivent avoir exactement la même résolution.")

    nb_cols = gt.shape[1]

    # ==========================================
    # Calcul des erreurs globales pour la similarité
    # ==========================================
    # Erreur totale de l'image de base (non corrigée) par rapport au 100%
    diff_base_sq = ((base - gt) ** 2).sum(axis=0)
    ssd_base = diff_base_sq.sum()

    # Erreur totale de l'image évaluée (corrigée) par rapport au 100%
    residu_sq = ((res - gt) ** 2).sum(axis=0)
    ssd_eval = residu_sq.sum()

    # Calcul du niveau de similarité (0% = image base, 100% = image parfaite)
    if ssd_base > 0:
        similarite = 100.0 * (1.0 - (ssd_eval / ssd_base))
    else:
        similarite = 100.0  # Si l'image de base n'avait aucun défaut dès le départ

    # ==========================================
    # Identification des colonnes défectueuses
    # ==========================================
    if json_path and os.path.exists(json_path):
        true_def = process_json(read_json(json_path))
    else:
        true_def = np.flatnonzero(diff_base_sq)

    true_mask = np.isin(np.arange(nb_cols), true_def)

    # ==========================================
    # Évaluation de la détection (res)
    # ==========================================
    detected = np.flatnonzero(residu_sq)

    # Calcul des TP, FP, FN
    tp = np.intersect1d(detected, true_def, assume_unique=True).size
    fp = np.setdiff1d(detected, true_def, assume_unique=True).size
    fn = np.setdiff1d(true_def, detected, assume_unique=True).size

    mask_def = true_mask
    mask_ok  = ~true_mask

    sq_def = (residu_sq[mask_def]).sum()
    cnt_def = mask_def.sum() * gt.shape[0]

    sq_ok  = (residu_sq[mask_ok]).sum()
    cnt_ok = mask_ok.sum() * gt.shape[0]

    nb_cols_f = float(nb_cols)
    tp_norm = tp / nb_cols_f
    fp_norm = fp / nb_cols_f
    fn_norm = fn / nb_cols_f

    prec = tp_norm / (tp_norm + fp_norm) if (tp_norm + fp_norm) else 0.0
    rec  = tp_norm / (tp_norm + fn_norm) if (tp_norm + fn_norm) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    
    rmse_def = np.sqrt(sq_def / cnt_def) if cnt_def else 0.0
    rmse_ok  = np.sqrt(sq_ok  / cnt_ok ) if cnt_ok  else 0.0

    # Retourner les résultats (j'ai enlevé les anciens RMSE normalisés qui ne te servent plus)
    return {
        "TP_norm": round(tp_norm, 6),
        "FP_norm": round(fp_norm, 6),
        "FN_norm": round(fn_norm, 6),
        "Precision": round(prec, 6),
        "Recall": round(rec, 6),
        "F1_Score": round(f1, 6),
        "RMSE_def": round(rmse_def, 6),
        "RMSE_ok": round(rmse_ok, 6),
        "Similarite_GT_pct": round(similarite, 2) # Arrondi à 2 décimales pour faire "pourcentage"
    }

def print_metrics_table(metrics):
    """
    Affiche un dictionnaire de métriques sous la forme d'un tableau formaté dans la console.
    """
    col_metrique = 20
    col_valeur = 12
    ligne_separatrice = "+" + "-" * (col_metrique + 2) + "+" + "-" * (col_valeur + 2) + "+"
    
    print("\nRÉSULTATS DE L'ÉVALUATION")
    print(ligne_separatrice)
    print(f"| {'Métrique'.ljust(col_metrique)} | {'Valeur'.rjust(col_valeur)} |")
    print(ligne_separatrice)
    
    for key, value in metrics.items():
        if key == "Similarite_GT_pct":
            print(ligne_separatrice)
            formatted_value = f"{value:.2f} %"
        else:
            formatted_value = f"{value:.6f}" if isinstance(value, float) else str(value)
            
        print(f"| {key.ljust(col_metrique)} | {formatted_value.rjust(col_valeur)} |")
        
    print(ligne_separatrice)

In [18]:
if __name__ == "__main__":
    resultats = evaluate_single_image(
        img_100_path=r"C:\Users\eliot\Desktop\Obsidian Vault\03. Cours\S2\Data_Challenge\train\VGA\sequence_3\low dyn\frame_0000.png",
        img_0_path=r"C:\Users\eliot\Desktop\Obsidian Vault\03. Cours\S2\Data_Challenge\train\VGA\sequence_3\low dyn with columns 1\frame_0000.png",
        img_eval_path=r"C:\Users\eliot\Desktop\Obsidian Vault\03. Cours\S2\Data_Challenge\train\VGA\sequence_3\low dyn with columns 1\frame_0000.png"
    )

    print(f"\nNiveau de similarité avec l'image parfaite : {resultats['Similarite_GT_pct']} %")
    print_metrics_table(resultats)


Niveau de similarité avec l'image parfaite : 0.0 %

RÉSULTATS DE L'ÉVALUATION
+----------------------+--------------+
| Métrique             |       Valeur |
+----------------------+--------------+
| TP_norm              |     0.004687 |
| FP_norm              |     0.000000 |
| FN_norm              |     0.000000 |
| Precision            |     1.000000 |
| Recall               |     1.000000 |
| F1_Score             |     1.000000 |
| RMSE_def             |    54.750951 |
| RMSE_ok              |     0.000000 |
+----------------------+--------------+
| Similarite_GT_pct    |       0.00 % |
+----------------------+--------------+
